<a href="https://colab.research.google.com/github/xc308/Dataset_preparation_Fine_Tuning/blob/main/Fine_tuning_all_parameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**All-parameters-fine-tuning**

- continue training your transformer on the new task by performing additional steps of gradient descent.

- can build specialized model with fewer training iterations

    - Loading a pre-trained model,
    - loading a new dataset for a new task,
    - preparing the new dataset,
    - fine-tuning the pre-trained model,
    - and evaluation of the model.

In [1]:
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For setting Keras configuration variables.

# Provides configuration for Keras.
os.environ["KERAS_BACKEND"] = "jax"

from textwrap import fill # For wrapping and formatting plain text.
from urllib.request import urlretrieve # For loading a model from a URL.

import keras # For defining and training the model.
import pandas as pd # For loading the dataset.
import tensorflow as tf # For shuffling the dataset.

# For displaying more readable error messages.
from IPython.display import display, HTML
from ai_foundations import training # For training your model.
from ai_foundations import generation # For prompting your model.
# For loading the tokenizer.
from ai_foundations.tokenization.bpe_tokenizer import BPEWordTokenizer
from ai_foundations import formatting # For formatting the training data.

keras.utils.set_random_seed(812) # For making the training reproducible.

  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-p8u64ovx
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-p8u64ovx
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 524d6114bbce631dafc00ba3496607a0bc60c804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Load Tokenizer**

- use the same tokenizer that was used when the model was originally trained.


In [2]:
url = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_tokenizer_3000.pkl"

tokenizer = BPEWordTokenizer.from_url(url)

Loaded pretrained tokenizer with vocabulary size 3,149.


**Load the pre-trained model**

-  first defining a Keras model with the same hyperparameters as the original pre-trained model
- and then using the model.load_weights method that sets all weights of the model to the final parameters of the pre-trained model.

In [3]:
model_url = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_1000ep.weights.h5"
model_filename = "africa_galore_qa_1000ep.weights.h5"
urlretrieve(model_url, model_filename)

# Define the model.
model = training.create_model(
    max_length=399,
    vocabulary_size=tokenizer.vocabulary_size,
)

model.load_weights(model_filename)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 2 variables whereas the saved optimizer has 40 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


**Load and prepare the data**

- # Load the question-answer dataset.


In [4]:
africa_galore_qa = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_v2.json"
)

africa_galore_qa.head(2)

,category,name,question,answer
0,Textile,Kente Cloth,What is Kente Cloth?,The vibrant colors and intricate patterns of K...
1,Textile,Bogolanfini (Mud Cloth),What is Bogolanfini (Mud Cloth)?,"Bogolanfini, or mud cloth, from Mali, is a tex..."


In [5]:
encoded_questions = [] # List of formatted and tokenized questions.
encoded_answers = [] # List of formatted and tokenized answers.

# Iterate through each item in the dataset.
for idx, row in africa_galore_qa.iterrows():

    # Run the format_qa function from the previous lab to format the question
    # and the answer.
    question, answer = formatting.format_qa(row) # format q and a

    # Tokenize the question and answer to be used as an input
    tokenized_q = tokenizer.encode(question)
    tokenized_a = tokenizer.encode(answer)

    # Append the tokenized question to the list of all tokenized questions.
    encoded_questions.append(tokenized_q)
    encoded_answers.append(tokenized_a)

In [6]:
# Print the number of questions.
print(f"Number of questions: {len(encoded_questions)}")

# Raise an error if the length of the answers is not the same as the length of
# the questions.
assert(len(encoded_questions) == len(encoded_answers))

# Print the first 10 tokens of the first two questions.
print() # Add your code here.

# Print the first 10 tokens of the first two answers.
print() # Add your code here.

Number of questions: 130




In [6]:
print(f"Number of questions: {len(encoded_questions)}")

Number of questions: 130


In [7]:
assert(len(encoded_answers) == len(encoded_questions))

In [14]:
print(encoded_questions[0][0:10])
print(encoded_questions[1][0:10])
print(encoded_questions[2][0:10])

[36, 2590, 2696, 151, 1926, 151, 2844, 1873, 39, 2900]
[36, 2590, 2696, 151, 1926, 151, 2844, 1873, 39, 2900]
[36, 2590, 2696, 151, 1926, 151, 2844, 1873, 39, 2900]


In [16]:
print(encoded_answers[0][0:10])
print(encoded_answers[1][0:10])

[36, 2590, 2696, 151, 1926, 151, 2844, 1873, 39, 1831]
[36, 2590, 2696, 151, 1926, 151, 2844, 1873, 39, 1831]


- When training a language model:

    - language models are trained by defining a sequence of input tokens and a sequence of target tokens.

    - The target tokens are a shifted version of the input tokens such that every element in the target is shifted by one position to the left.

    - During training, the model then learns to predict the target token at every position  𝑖  from all input tokens up to position  𝑖 .

    - The parameters are updated according to the loss of each target token, so that during the next iteration of training, the model makes better predictions for each token in the training data.

Fine-tuning is the same.

- First, also have to prepare the data as a sequence of input and target tokens.

- For question-answer tasks, such as the flashcard generation task, you therefore have to turn the question-answer pair into a single sequence of tokens. You can do this by concatenating the question and the answer: <start_of_turn>user What is Jollof rice?<end_of_turn><start_of_turn>model Category: Food. \nJollof rice is a...

In [8]:
inputs = ["<start_of_turn>", "user", "What", "is", "Jollof", "rice", "?", "<end_of_turn>", "<start_of_turn>", "model", "Category", ":", "Food", "\n", "Jollof", "rice", "is", ...]
targets = ["user", "What", "is", "Jollof", "rice", "?", "<end_of_turn>", "<start_of_turn>", "model", "Category", ":", "Food", "\n", "Jollof", "rice", "is", ...]


To avoid teaching the model unnecessary skills, limit the computation of the loss function to the tokens in the response.


means that the model still learns to base the response on the prompt, since it can attend to the tokens in the prompt when predicting tokens in the answer. In this case the weights are only updated such that the model performs better at the task of interest


to ignore the tokens in the question from the loss computation is to replace them with the special <PAD> token in the target sequence.

 loss function is generally implemented such that it ignores the predictions at all positions where the target token is <PAD>

In [9]:
# Inputs are the same as before.
inputs = ["<start_of_turn>", "user", "What", "is", "Jollof", "rice", "?", "<end_of_turn>", "<start_of_turn>", "model", "Category", ":", "Food", "\n", "Jollof", "rice", "is", ...]
targets = ["<PAD>", "<PAD>", "<PAD>", "<PAD>", "<PAD>", "<PAD>", "<PAD>", "<start_of_turn>", "model", "Category", ":", "Food", "\n", "Jollof", "rice", "is", ...]

In [10]:
encoded_inputs = []
encoded_targets = []
pad_token_id = tokenizer.pad_token_id
for idx in range(len(encoded_questions)):
    # Length of the question.
    len_q = len(encoded_questions[idx])
    # Concatenate a question and answer.
    concatenated_tokens = encoded_questions[idx] + encoded_answers[idx]
    # Concatenate q and a but with question masked with padding token.
    concatenated_masked = [pad_token_id] * len_q + encoded_answers[idx]
    # Add concatenated tokens to inputs, except the last token.
    encoded_inputs.append(concatenated_tokens[:-1])
    # Add concatenated tokens to outputs, shifted by 1 place.
    encoded_targets.append(concatenated_masked[1:])

In [19]:
encoded_inputs = []
encoded_targets = []

pad_token_id = tokenizer.pad_token_id
for idx in range(len(encoded_questions)):
    len_q = len(encoded_questions[idx])

    concatenated_tokens = encoded_questions[idx] + encoded_answers[idx] # inputs
    concatenated_masked = [pad_token_id] * len_q + encoded_answers[idx] # targets

    encoded_inputs.append(concatenated_tokens[:-1])
    encoded_targets.append(concatenated_masked[1:])

**Other data preparation**

- also need to pad and truncate individual examples so that you can group them into batches

- shuffling the data can help with more efficient training as it can avoid that very similar examples appear in the same batch during training

- when fine-tuning, you have to adhere to the maximum length of the pre-trained model, therefore extracting the maximum sequence length from the pre-trained model.



**Info: Batch size during fine-tuning**


- The batch size for fine-tuning is often set to a smaller value than during pre-training.

    - in part due to memory constraints, as all examples in a batch need to be stored on a GPU.

- Additionally, this often leads to better model performance.

    - This is because the gradient estimates from small batches tend to be noisier than from large batches
    - this noise can act as a form of regularization that makes it less likely that the model overfits

-

**perform padding, truncating, and batching and shuffling**

In [11]:
# Maximum length for an input text. input_shape[0] is the batch size.
max_input_length = model.input_shape[1]
max_output_length = model.output_shape[1]

# The input sequences are the padded questions.
# Use Keras for padding and truncating.
input_sequences = keras.preprocessing.sequence.pad_sequences(
    encoded_inputs,           # Data to be truncated.
    maxlen=max_input_length,  # Maximum length.
    padding="post",           # Pad at the end if necessary
    truncating="post",        # Truncate at end if necessary.
    value=tokenizer.pad_token_id # The ID of the pad token.
)

# The target sequences are the padded answers.
target_sequences = keras.preprocessing.sequence.pad_sequences(
    encoded_targets,
    maxlen=max_output_length,
    padding="post",
    truncating="post",
    value=tokenizer.pad_token_id # The ID of the pad token
)

# Create TensorFlow dataset to prepare sequences.
tf_dataset = tf.data.Dataset.from_tensor_slices(
    (input_sequences, target_sequences)
)

# Randomly shuffle the dataset.
tf_dataset = tf_dataset.shuffle(buffer_size=len(input_sequences))

# Specify batch size.
batch_size = 4

# Create batches.
batches = tf_dataset.batch(batch_size)

**Fine-tuning the model**

- With data being prepared, can now fine-tune model on the flashcard generation task by continuing the original training process


- defining a model and loading its parameters  and then continuing training with the model.fit()

**Learning rate**

- When fine-tuning a model, the learning rate determines how much the weights are updated during each step of stochastic gradient descent.

- when fine-tuning a model, you want to retain the knowledge that has already been learned rather than making significant updates

- you generally want to use a smaller learning rate during fine-tuning than during pre-training

- A common learning rate for pre-training is 0.0001 (1e-4) and for fine-tuning, want to use a learning rate that is one or two orders of magnitude lower, so a value between 0.00001 (1e-5) and 0.000001 (1e-6) tends to be well-suited.

- If you choose higher values, then the learning process may end up being more susceptible to catastrophic forgetting.
    - If you fit the model parameters too closely to the patterns in the fine-tuning dataset, the model may "forget" what it has learned during pre-training

    - important to find the proper point to stop training on the new data

    - make sure that you stop training after the model has adjusted to the new task, but before the knowledge gained during pre-training is overwritten



In [12]:
learning_rate = 5e-5 # @param {type: "number"}
model.optimizer.learning_rate = learning_rate

**Number of epochs**

- The number of epochs that are needed for successful training strongly depends on
    - the size of the dataset: if dataset was ten times as big, one epoch of the big dataset would equal ten epochs of the small dataset; more diverse but also more expensive
     
    - the learning rate: the smaller the learning rate, the more training updates you will need
    -

In [13]:
num_epochs = 50 # @param {type: "number"}

**Callback function**

- allows you to monitor the progress of training and get a sense of what the model has learned after each epoch

- enables you to determine when to stop training and whether the setting of the learning rate is reasonable.

-

In [15]:
prompt = "<start_of_turn>user\nWhat is Jollof rice?<end_of_turn>\n"

prompt_ids = tokenizer.encode(prompt)
text_gen_callback = training.TextGenerator(
    max_tokens=30,
    start_tokens=prompt_ids,
    tokenizer=tokenizer,
)

training.TextGenerator(

)

**Train the model**

In [16]:
history = model.fit(
    x = batches,
    epochs = num_epochs,
    callbacks = [text_gen_callback] # List of keras.callbacks.Callback instances. List of callbacks to apply during training.
)


Epoch 1/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 800ms/step - loss: 5.6074Generated text:
 <start_of_turn>user What is Jollof rice?<end_of_turn> <wera_of_allit_g_of_e__ou_m_of_ou__ou_ou_ 

33/33 ━━━━━━━━━━━━━━━━━━━━ 36s 931ms/step - loss: 4.9701
Epoch 2/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 619ms/step - loss: 3.8368Generated text:
 <start_of_turn>user What is Jollof rice?<end_of_turn> <em> <start<start_turn>orange thread swana ennamed Blense's fast ingredients, 

33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 668ms/step - loss: 3.7661
Epoch 3/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 639ms/step - loss: 3.4238Generated text:
 <start_of_turn>user What is Jollof rice?<end_of_turn> <start_turn>model Category: chicken, or chicken and taris added next, made primar 

33/33 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - loss: 3.4111
Epoch 4/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 618ms/step - loss: 3.2393Generated text:
 <start_of_turn>user What is Jollof rice?<end_of_turn> <start_of_of_of_of_turn>a of fish fle_of_turtrou_turnal 

33/33 ━━━━━━━━━

**Test the model**

In [17]:
# @title Prompt your fine-tuned model
question = "Jollof rice is" #@param {type: "string"}
is_question = False # @param {type: "boolean"}
if is_question:
    prompt = "<start_of_turn>user\n" + question + "<end_of_turn>\n"
else:
    prompt = question
num_tokens_to_generate = 30 #@param {type: "number"}
generated_text, probs = generation.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="greedy" # To generate the highest probability generation.
)

print("Generated text: ", fill(generated_text, replace_whitespace=False))
print("\n")

Generated text:  Jollof rice is a major city in East Africa, with some areas a major
exposure for explosive lamb. It has a




In [18]:
# @title Prompt your fine-tuned model
question = "What is Mount Kilimanjaro?" #@param {type: "string"}
is_question = True # @param {type: "boolean"}
if is_question:
    prompt = "<start_of_turn>user\n" + question + "<end_of_turn>\n"
else:
    prompt = question
num_tokens_to_generate = 30 #@param {type: "number"}
generated_text, probs = generation.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="greedy" # To generate the highest probability generation.
)

print("Generated text: ", fill(generated_text, replace_whitespace=False))
print("\n")

Generated text:  <start_of_turn>user What is Mount Kilimanjaro?<end_of_turn>
<start_of_turn>model Category: Drink Mount Kenya is a self




In [19]:
# @title Prompt your fine-tuned model
question = "What is Mount Aconcagua?" #@param {type: "string"}
is_question = True # @param {type: "boolean"}
if is_question:
    prompt = "<start_of_turn>user\n" + question + "<end_of_turn>\n"
else:
    prompt = question
num_tokens_to_generate = 30 #@param {type: "number"}
generated_text, probs = generation.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="greedy" # To generate the highest probability generation.
)

print("Generated text: ", fill(generated_text, replace_whitespace=False))
print("\n")

Generated text:  <start_of_turn>user What is Mount Aconcagua?<end_of_turn>
<start_of_turn>model Category: Flora The Batik is a traditional E




the question about the South American mountain, Mount Aconcagua, did not yield a sensible response. This is because neither the pre-training data of the model (the Africa Galore dataset) nor the fine-tuning data contains any information on this mountain.